### Make a temporary space in /flash

In [1]:
import os, random, string

In [2]:
cwd = os.path.basename(os.getcwd())
tmpdir = os.path.join("/flash/ReiterU/tmp/", 'tomo_' + ''.join(random.choices(string.ascii_uppercase, k=12)))
print(tmpdir)
if not os.path.exists(tmpdir):
    os.makedirs(tmpdir)

/flash/ReiterU/tmp/tomo_BYPUINTFVGTO


### PARAMETERS

In [3]:
#### path to relevant files
rootdir = "/bucket/ReiterU/OLdev/experiments/20250522_squid_cal520"
recname = "rec3"
savedir = "/bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250522_rec3"

templatefile = os.path.join(rootdir, "2P", recname, "run2/mcorr_nonrigid/QC/QC_nonrigid/template.tif")
roifile = os.path.join(rootdir, "2P", recname, "ROI_yutaka_renamed.zip")
dfffile = os.path.join(rootdir, "2P", recname, "dff_ROI_yutaka/DFF.hdf5")
syncfile = os.path.join(rootdir, "sync", f"{recname}.tdms")

QI_THRESH = 0.25

### Import stuff

In [4]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy import signal
import h5py
import subprocess

In [5]:
import sys
sys.path.append("../../../")
from calfuncs.io import read_sync_ttl
from calfuncs.full_field_step import get_ett2, plot_chirp_eta, compute_qi

In [6]:
from matplotlib import font_manager
import matplotlib as mpl

font_manager.fontManager.addfont('/bucket/ReiterU/common/fonts/arial.ttf')
mpl.rcParams.update({
    'font.size': 7,
    'font.family': 'sans-serif',
    'font.sans-serif': "Arial",
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'xtick.major.size': 2,
    'xtick.minor.size': 2,
    'xtick.minor.width': 0.5,
    'ytick.major.width': 0.5,
    'ytick.minor.width': 0.5,
    'ytick.major.size': 2,
    'ytick.minor.size': 2,
})

### Load TDMS file

In [7]:
sync_ttl = read_sync_ttl(syncfile)
twop_ttl = sync_ttl["twop_ttl"]
twop_re = sync_ttl["twop_re"]
vis_re = sync_ttl["vis_re"]
print(twop_ttl.shape, twop_re.shape, vis_re.shape)

(717000,) (10912,) (60,)


### Load DF/F

In [8]:
with h5py.File(dfffile, "r") as hf:
    F = hf["F"][:]
    dff = hf["dff"][:]
print(F.shape) # F is sampled at 1 kH

(248, 717000)


### Chirp

In [9]:
E = []
for nid in np.arange(0, F.shape[0]):
    ett1 = get_ett2(F, nid, vis_re[0::6], window=[-2000, 19000], baseline_window=[-4000, 70000], subsampling=50)
    ett1["session"] = 0
    ett2 = get_ett2(F, nid, vis_re[2::6], window=[-2000, 19000], baseline_window=[-27000, 43000], subsampling=50)
    ett2["session"] = 1
    ett3 = get_ett2(F, nid, vis_re[4::6], window=[-2000, 19000], baseline_window=[-51000, 19000], subsampling=50)
    ett3["session"] = 2
    ett = pd.concat([ett1, ett2, ett3])
    ett["neuron"] = nid
    ett = ett.set_index(["neuron", "session", "trial", "time"])
    E.append(ett)
E = pd.concat(E)

In [ ]:
dt = 1e-2
stimulus = np.zeros(int(21.0/dt), dtype=np.float32)
stimulus[0:int(2.0/dt)] = 0
stimulus[int(2.0/dt):int(6.0/dt)] = 1
stimulus[int(6.0/dt):int(10.0/dt)] = 0
stimulus[int(10.0/dt):int(12.0/dt)] = 0.5
stimulus[int(12.0/dt):int(20.0/dt)] = 0.5 + 0.5 * signal.chirp(np.linspace(0, 8, int(20.0/dt)-int(12.0/dt)), f0=0.5, f1=10, t1=8, method='logarithmic', phi=90)
stimulus[int(20.0/dt):int(21.0/dt)] = 0

for nid in np.arange(0, dff.shape[0]):
    qi = compute_qi(E, nid)
    print("Neuron:", nid, "QI:", qi)
    savename = os.path.join(tmpdir, f"ETA_{nid:03d}.png")
    plot_chirp_eta(E, nid, stimulus, vmin=-2, vmax=3,  cmap_name="bone_r", figsize=(5,2.0), show_plot=True, figsavename=savename)

### Compute trial-average

In [11]:
R = []
neurons = E.index.get_level_values("neuron").unique()

for n in neurons:
    M = []
    df2 = E.xs(n, level="neuron")
    for i in range(3):
        df3 = df2.xs(i, level="session")
        mat = df3.reset_index().pivot(index="trial", columns="time", values="y").to_numpy()
        M.append(mat)
    M = np.hstack(M)
    M = M[~np.all(np.isnan(M), axis=1)] # remove trials with NaN values
    qi = compute_qi(E, n)
    y_mean = np.nanmean(M, axis=0)
    row = {
        "recording": 0,
        "neuron": n,
        "qi": qi,
    }
    for j, value in enumerate(y_mean):
        row[f"time_{j}"] = value

    R.append(row)

R = pd.DataFrame(R)

In [12]:
display(R)

,recording,neuron,qi,time_0,time_1,time_2,time_3,time_4,time_5,time_6,...,time_1250,time_1251,time_1252,time_1253,time_1254,time_1255,time_1256,time_1257,time_1258,time_1259
0,0,0,0.528743,-0.695040,-0.628646,-0.529582,-0.522205,-0.895225,-0.875878,-1.024086,...,-0.174630,0.003019,0.167194,-0.205736,-0.087147,-0.228439,-0.123187,-0.306020,-0.282387,-0.343976
1,0,1,0.410118,0.274520,0.324279,0.005827,0.044786,0.482410,0.658077,0.457329,...,-0.348171,-0.160571,-0.584498,-0.960795,-0.599307,-0.373107,-0.450208,-0.655112,-0.371987,-0.240413
2,0,2,0.694902,0.140078,0.170821,-0.045185,-0.019525,0.138607,0.048706,-0.024375,...,-0.217646,-0.032712,-0.268689,-0.069820,0.226245,-0.020367,-0.131970,0.334467,0.359025,0.305052
3,0,3,0.858993,-0.792745,-0.845572,-0.904623,-0.886388,-0.806948,-0.751978,-0.757128,...,0.524136,0.479441,0.503971,0.583264,0.617018,0.721766,0.677003,0.726810,0.745708,0.829642
4,0,4,0.566046,-0.142874,-0.120623,0.095829,-0.255364,-0.290718,-0.153089,-0.095619,...,0.409204,0.491882,0.651977,0.502640,0.313747,0.319211,0.423198,0.530166,0.551350,0.718174
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243,0,243,0.793841,0.981528,0.730736,0.577102,0.659060,0.856712,0.652413,0.810701,...,-0.501737,-0.528515,-0.368494,-0.308503,-0.324965,-0.505007,-0.705178,-0.593609,-0.565772,-0.561358
244,0,244,0.275686,0.566814,0.872461,0.739290,0.045465,0.166549,0.072119,0.121159,...,0.138205,-0.037690,-0.223342,0.265267,0.416145,0.267679,0.230837,0.415981,0.634294,0.302843
245,0,245,0.850508,-0.576816,-0.522058,-0.471030,-0.636829,-0.693659,-0.836083,-0.774822,...,0.401697,0.456299,0.587097,0.743977,0.750437,0.879612,0.925039,0.807301,0.825511,0.843471
246,0,246,0.228386,-0.719940,-1.001742,-0.611929,-0.045221,-0.367229,-0.556999,-0.607992,...,-0.262271,-0.752324,-0.817930,-0.450104,-0.386749,-0.640777,-0.640633,-0.188466,-0.281075,-0.491129


In [13]:
with pd.HDFStore(os.path.join(tmpdir, "response_table.hdf5"), mode='w') as store:
    store.put('E', E)
    store.put('responses', R)

### Copy data to bucket

In [14]:
cmd1 = f'ssh deigo "mkdir -p {savedir}"'
cmd2 = f'ssh deigo "cp -r {tmpdir}/* {savedir}"'
print(cmd1)
print(cmd2)

subprocess.run(cmd1, shell=True, check=True)
subprocess.run(cmd2, shell=True, check=True)

ssh deigo "mkdir -p /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250522_rec3"
ssh deigo "cp -r /flash/ReiterU/tmp/tomo_BYPUINTFVGTO/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250522_rec3"


Offending key for IP in /home/t/tomoyuki-mano1/.ssh/known_hosts:1
Matching host key in /home/t/tomoyuki-mano1/.ssh/known_hosts:6


CompletedProcess(args='ssh deigo "cp -r /flash/ReiterU/tmp/tomo_BYPUINTFVGTO/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250522_rec3"', returncode=0)